<a href="https://colab.research.google.com/github/BandanaSingha24/01.-Python-Foundation-For-Bioinformatics/blob/main/NCBI_Sequence_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.7 MB/s eta 0:00:00


In [3]:
from Bio import SeqIO
for record in SeqIO.parse("/content/sequence.fasta", "fasta"):
  print("Gene ID:", record.id)
  print("length:", len(record.seq))
  print("Sequence:", record.seq[:50])

Gene ID: M55422.1
length: 2873
Sequence: GACTAATGGCCCTAACTAAGTATTAACCCCTGCCAAGAAAAGAGCTACTT


In [9]:
from Bio import SeqIO
for record in SeqIO.parse("/content/sequence.fasta","fasta"):
     gc_count = record.seq.count('G') + record.seq.count('C')
     total_length = len(record.seq)
     if total_length > 0:
         gc_percentage = (gc_count / total_length) * 100
     else:
         gc_percentage = 0.0
     print("Gene ID:",record.id)
     print("GC Content Percentage:",round(gc_percentage,2),"%")

Gene ID: M55422.1
GC Content Percentage: 38.6 %


In [11]:
mrna=record.seq.transcribe()
protein=mrna.translate()
print("Protein Sequence:",protein)

Protein Sequence: D*WP*LSINPCQEKSYFLLK*MKIVMLSLKLYL*KASKGGMKQEI*KEKQVKGKQVLS*PV*LTPKSCWSYDNYLQGQAGAPKEGSRSRDEKNKFSLSVSLFEILSP*HYSLFCSHNYFCNYFCKSVKIL*VLVFLSVAWQGHKTCLSKVGSCCKSCCKTCQRYD*LPLFCFCKTAFSPRRFCAKNPTCPCLMHV*KSSPVFVPGSAFGC*SAGPVAT*IKPSCCTQ*SLRPPGSHNI*AFLFSAT*PASVIDFLNNRHSDWMRWYLTVVFLCIFLIINNVNFVFMCLSATCMSSFEKYLFMFLLTF*LGFWFSFCCCCCFLRWSLAVSPGWSAVA*SRLTATSTSQVQMILLLQSPE*LGSQGPLTFMDVAIEFSLEEWQCLDTAQQNLYRHVM*ENYRNLVFLGEGIAVSKADLITCLEQGKEPWNMKRHEMVAKHLVMFYYFAQHLWPEQNIRDSFQKVTLRRYRKCGYENLQLRKGCKSVVECKQHKGDYSGLNQCLKTTLSKIFQCNKYVEVFHKISNSNRHKMRHTENKHFKCKECRKTFCMLSHLTQHKRIQTRVNFYKCEAYGRAFNWSSTLNKHKRIHTGEKPYKCKECGKAFNQTSHLIRHKRIHTEEKPYKCEECGKAFNQSSTLTTHNIIHTGEIPYKCEKCVRAFNQASKLTEHKLIHTGEKRYECEECGKAFNRSSKLTEHKYIHTGEKLYKCEECGKAFNQSSTLTTHKRIHSGEKPYKCEECGKAFKQFSNLTDHKKIHTGEKPYKCEECGKAFNQLSNLTRHKVIHTGEKPYKCGECGKAFNQSSALNTHKIIHTGENPHKCRESGKVFHLSSKLSTCKKIHTGEKLYKCEECGKAFN*SSTLIGHKRIHTGEKPYKCEECGKAFNQSSTLTTHKIIHTEEKQYKCDECGKAST*SSKLIEHKKIHTGEKPYKCEECGKTSNHFSTFTKHKMIHTGEKLH


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [13]:
protein_clean=mrna.translate(to_stop=True)
print("Length:",len(protein_clean))
print("Sequence:",protein_clean)

Length: 1
Sequence: D


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [15]:
full_protein=str(mrna.translate())
if "M" in full_protein:
  real_protein=full_protein[full_protein.index("M"):]
  real_protein_clean=real_protein.split("*")[0]
  print("Length of real protein:",len(real_protein_clean))
  print("Real Protein Sequence:\n",real_protein_clean)
else:
  print("No valid protein starting with M found.")

Length of real protein: 12
Real Protein Sequence:
 MKIVMLSLKLYL


/usr/local/lib/python3.12/dist-packages/Bio/Seq.py:2874: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


In [17]:
from Bio.Blast import NCBIWWW
print('Starting NCBI BLAST search...')
result_handle=NCBIWWW.qblast("blastp","nr","MKIVMLSLKLYL")
print("BLAST search completed successfully!")

Starting NCBI BLAST search...
BLAST search completed successfully!


In [23]:
from Bio.Blast import NCBIXML

try:
    blast_reccord=NCBIXML.read(result_handle)
    for alignment in blast_reccord.alignments[:3]:
       for hsp in alignment.hsps:
        print("Protein Name:",alignment.title)
        print ("E-value:",hsp.expect)
        print("-"*50)
except ValueError as e:
    print(f"Error parsing BLAST results: {e}")
    print("It appears the BLAST search yielded no significant matches or the result handle was empty.")
    print("This can happen if the query sequence is too short, or no similar sequences are found in the database.")

Error parsing BLAST results: Your XML file was empty
It appears the BLAST search yielded no significant matches or the result handle was empty.
This can happen if the query sequence is too short, or no similar sequences are found in the database.


In [25]:
from Bio.Blast import NCBIWWW
from Bio.Blast import NCBIXML

query_seq = "MKIVMLSLKLYL"

print("Starting custom BLAST for short sequences... Please wait.")
result_handle = NCBIWWW.qblast("blastp", "nr", query_seq, expect=20000, short_query=True)
print("BLAST search completed!")

print("Parsing results...")
try:
    blast_record = NCBIXML.read(result_handle)
    if not blast_record.alignments:
        print("No significant matches found even with short query parameters.")
    else:
        print("=" * 50)
        print("MATCHING PROTEINS FOUND:")
        print("=" * 50)
        for alignment in blast_record.alignments[:3]:
            for hsp in alignment.hsps:
                print("Protein Name:", alignment.title)
                print("E-value:", hsp.expect)
                print("-" * 50)
except Exception as e:
    print("Error during parsing:", e)

Starting custom BLAST for short sequences... Please wait.
BLAST search completed!
Parsing results...
MATCHING PROTEINS FOUND:
Protein Name: gb|KAN5666442.1| FACT complex subunit spt16 [Trichinella nativa]
E-value: 42.177
--------------------------------------------------
Protein Name: gb|KRZ57053.1| Prestin [Trichinella nativa]
E-value: 42.1977
--------------------------------------------------
Protein Name: gb|KRZ57052.1| Prestin [Trichinella nativa]
E-value: 42.2021
--------------------------------------------------
